## Load prerequisite modules and data

* To run this as a CLI script
```jupyter nbconvert report_rnn_reps_ssh.ipynb --to python; python -u report_rnn_reps_ssh.py```


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation
import numpy as np

print("numpy version:", np.__version__)
import sys
import argparse
import os
import glob
import tqdm

In [ ]:
BASE_PATH = "../../../../../srv/marl/satsingh/marl_fish/"

# SUBFOLDER="/rmappo-MultiAgentFishEnv-114/outputs/"
# FLAT_PKL_FILE="MAFish_neural_20241013_202859_gQi2w14j_flattened.pkl"

SUBFOLDER="/20241124_004407/outputs/"
FLAT_PKL_FILE="MAFish_neural_20241124_004407_v3Tj9CEU_agg_flattened.pkl"

full_path = BASE_PATH + SUBFOLDER + FLAT_PKL_FILE

print(f"Using .pkl file: {full_path}")
dff = pd.read_pickle(full_path)

print("dff.shape", dff.shape)
print("dff.columns", dff.columns)
# dff.head()

## Agent-specific PCAs

In [ ]:
# Plot PCA of RNN states (2D and 3D) -- Not colored by any specific variable
agent_ids = dff["agent_id"].unique().tolist()
print(agent_ids)
for agent_id in agent_ids[:1]:
    print(f"Agent ID: {agent_id}")

    dff_agent = dff[dff["agent_id"] == agent_id]
    rnn_states = dff_agent["rnn_states"].tolist()

    # Perform PCA
    pca = PCA()
    rnn_states_pca = pca.fit_transform(rnn_states)
    cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
    plt.figure(figsize=(6, 4))
    plt.plot(cumulative_variance, marker="o")
    plt.axhline(y=0.90, color="r", linestyle="--", label="90% variance explained")
    pc90_index = np.where(cumulative_variance >= 0.90)[0][0]
    plt.axvline(x=pc90_index, color="b", linestyle="--", label=f"PC{pc90_index+1}")
    plt.title(f"Cumulative Variance Explained for Agent: {agent_id}")
    plt.xlabel("Principal Component Index")
    plt.ylabel("Cumulative Variance Explained")
    plt.legend()
    plt.show()

    # Perform PCA
    pca_2d = PCA(n_components=2)
    pca_3d = PCA(n_components=3)

    rnn_states_2d = pca_2d.fit_transform(rnn_states)
    rnn_states_3d = pca_3d.fit_transform(rnn_states)

    # 2D PCA plot
    plt.figure(figsize=(5, 4))
    plt.scatter(
        rnn_states_2d[:, 0],
        rnn_states_2d[:, 1],
        c=dff_agent["agent_id"],
        cmap="viridis",
        alpha=0.5,
        s=4,
    )
    plt.title(f"Agent: {agent_id}")
    plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.2f}% var)")
    plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.2f}% var)")
    plt.colorbar()
    plt.show()

    # 3D PCA plot
    fig = plt.figure(figsize=(8, 5))
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(
        rnn_states_3d[:, 0],
        rnn_states_3d[:, 1],
        rnn_states_3d[:, 2],
        c=dff_agent["agent_id"],
        cmap="viridis",
        alpha=0.5,
        s=4,
    )
    ax.set_title(f"Agent: {agent_id}")
    ax.set_xlabel(f"PC1 ({pca_3d.explained_variance_ratio_[0]*100:.2f}% var)")
    ax.set_ylabel(f"PC2 ({pca_3d.explained_variance_ratio_[1]*100:.2f}% var)")
    ax.set_zlabel(f"PC3 ({pca_3d.explained_variance_ratio_[2]*100:.2f}% var)")

    # fig.colorbar(scatter, label='Agent ID')
    cbar = fig.colorbar(scatter, pad=0.1)
    cbar.set_label("")
    plt.show()